# Probe Runner

This notebook drives `Probe.py` to train linear and non-linear classifiers on
every (model, dataset) hidden-state artifact produced by the extraction stage.
It is the **second** stage of the pipeline:

    master_dataset.py  ->  datasets/<name>/processed/<name>_clean.csv
    Extraction.py      ->  hidden_states/<slug>/<dataset>/*.npy
    Probe.py           ->  interEx/<slug>/<dataset>/<trial>/*.csv

## How to read this notebook

Each code cell is preceded by a markdown cell that says what it does and why
it comes at this position. Every code cell prints a **banner**, its
**inputs**, its **work**, and its **outputs**, so a failure is reported where
it happens.

## Flow

| # | Cell | Purpose |
|---|---|---|
| 2 | Bootstrap | imports, theme, path checks, system fingerprint |
| 4 | Datasets  | discover + validate processed CSVs |
| 6 | Contracts | resolve per-dataset contracts, define probe specs |
| 8 | Discovery | helpers to walk hidden_states/ and interEx/ |
| 10| Pre-flight| contract + probe sanity checks, target list |
| 12| Run       | call `Probe.run_matrix` |
| 14| Summary   | best-layer table, plots, per-pair layer curves |

## Conventions

- All paths come from `_shared.py`. Nothing hardcoded.
- `VERBOSITY` (cell 2) controls notebook chatter, not `Probe.py`'s.
- A probe trial folder is keyed by a hash of the full config (contract +
  probes + split). Changing any of these creates a new folder and the run
  starts over. Do not edit cell 6 while a matrix is running.


In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — BOOTSTRAP
#
# Verifies every path the probe pipeline will read from or write to. If a
# required path is missing, this cell raises immediately — before any
# artifact is loaded or any probe is fitted.
# ═══════════════════════════════════════════════════════════════════════

import importlib, json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# ── Theme ──────────────────────────────────────────────────────────────
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important; color: #d4d4d4 !important;
    }
    h2, h3, h4 { color: #4fc3f7 !important; border-bottom: 2px solid #3498db !important; }
    b, strong { color: #f48fb1 !important; }
    .highlight { background-color: #2d2d2d !important; padding: 10px;
                 border-left: 4px solid #3498db; margin: 4px 0; color: #d4d4d4; }
    code { background-color: #333 !important; color: #ffcc80 !important;
           padding: 2px 4px; border-radius: 4px; }
    .dataframe { background-color: #2d2d2d !important; color: #d4d4d4 !important; }
</style>
"""))

def display_title(t): display(HTML(f"<h2>{t}</h2>"))
def display_info(m):  display(HTML(f"<div class='highlight'>{m}</div>"))

# ── Verbosity: one control for this notebook only ──────────────────────
VERBOSITY = "info"
_ORDER = {"debug": 0, "info": 1, "quiet": 2}
def _say(msg, level="info"):
    if _ORDER[level] >= _ORDER[VERBOSITY]:
        print(msg)

def _banner(title):
    print(f"\n{'-' * 72}\n{title}\n{'-' * 72}")

# ── Pipeline modules ───────────────────────────────────────────────────
import _shared
importlib.reload(_shared)
import Extraction as EX
importlib.reload(EX)
import Probe as probe
importlib.reload(probe)

from _shared import (
    AMIRALI_MOUNT, DATASETS_ROOT,
    HIDDEN_STATES_ROOT, INTEREX_ROOT, PROBE_ROOT,
    model_slug,
)

_banner("STEP 1 / 6 — Path verification")

REQUIRED = {
    "Drive mount":       AMIRALI_MOUNT,
    "Datasets root":     DATASETS_ROOT,
    "Hidden states":     HIDDEN_STATES_ROOT,
}
OPTIONAL = {
    "InterEx (probes)":  INTEREX_ROOT,
}
for label, p in REQUIRED.items():
    assert p.is_dir(), f"Required path missing: {p}"
    _say(f"  [OK]  {label:18s}  {p}")
for label, p in OPTIONAL.items():
    mark = "[OK]" if p.is_dir() else "[--]"
    note = "" if p.is_dir() else "  (created on first probe run)"
    _say(f"  {mark}  {label:18s}  {p}{note}")

_banner("STEP 1 / 6 — System")

import platform, torch
_say(f"  Platform       : {platform.platform()}")
_say(f"  Python         : {platform.python_version()}")
_say(f"  PyTorch        : {torch.__version__}")
_say(f"  CPU threads    : {torch.get_num_threads()}")
try:
    import psutil
    vm = psutil.virtual_memory()
    _say(f"  RAM total      : {vm.total / 1e9:.2f} GB")
    _say(f"  RAM available  : {vm.available / 1e9:.2f} GB")
except ImportError:
    _say("  RAM info       : psutil not installed (pip install psutil)")
_say(f"  Probe device   : {probe.choose_device()}")

print("\nOK Bootstrap complete. Next cell discovers processed datasets.")



------------------------------------------------------------------------
STEP 1 / 6 — Path verification
------------------------------------------------------------------------
  [OK]  Drive mount         /Volumes/Amirali
  [OK]  Datasets root       /Volumes/Amirali/datasets
  [OK]  Hidden states       /Volumes/Amirali/hidden_states
  [OK]  InterEx (probes)    /Volumes/Amirali/interEx

------------------------------------------------------------------------
STEP 1 / 6 — System
------------------------------------------------------------------------
  Platform       : macOS-14.6.1-x86_64-i386-64bit
  Python         : 3.12.0
  PyTorch        : 2.2.2
  CPU threads    : 4
  RAM total      : 8.59 GB
  RAM available  : 2.37 GB
  Probe device   : cpu

OK Bootstrap complete. Next cell discovers processed datasets.


## Processed datasets

`master_dataset.py` writes one CSV per dataset at
`datasets/<name>/processed/<name>_clean.csv`. `EX.discover_processed_datasets`
walks that root and returns two dicts:

- `DATASETS`   — `{name: DataFrame}`
- `CSV_HASHES` — `{name: sha256}`

Both are needed: the DataFrames carry the target labels the probe will
learn from, and the hashes let us verify (later, in `Probe.py`) that the
hidden states were extracted from the exact same CSV.

### Contract enforced by the next cell

| Check | Why |
|---|---|
| Columns are exactly `clean_text, label, sentiment_score` | Extraction and probing both assume them |
| Index is a clean `RangeIndex 0..N-1` | Row `i` in CSV maps to row `i` in `hidden_states.npy` |
| Every label is a non-empty Python list | The probe's label adapters assume it |
| No null `clean_text` rows | Extraction would have dropped them and desynced |

The next cell raises `AssertionError` if any dataset violates the contract.
A bad dataset must not reach the probe stage.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — DATASET DISCOVERY & CONTRACT CHECK
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 2 / 6 — Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(show_info=True)
assert DATASETS, "No processed datasets discovered."

_failures = []
for name, df in DATASETS.items():
    if list(df.columns) != list(EX.PROCESSED_COLUMNS):
        _failures.append(f"{name}: bad columns {list(df.columns)}")
        continue
    if not (df.index.is_unique and df.index[0] == 0
            and df.index[-1] == len(df) - 1):
        _failures.append(f"{name}: index is not a clean RangeIndex")
    if not df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all():
        _failures.append(f"{name}: some labels are not non-empty lists")
    nulls = int(df["clean_text"].isna().sum())
    if nulls:
        _failures.append(f"{name}: {nulls} null clean_text rows")

if _failures:
    print()
    for f in _failures:
        print(f"  X {f}")
    raise AssertionError(f"{len(_failures)} dataset(s) failed the contract")

_banner("STEP 2 / 6 — Discovery summary")
summary = pd.DataFrame([
    {
        "name":         name,
        "rows":         f"{len(df):,}",
        "n_classes":    str(len(set(x for row in df["label"] for x in row))),
        "sample_label": str(df["label"].iloc[0]),
        "csv_sha256":   CSV_HASHES[name][:16] + "...",
    }
    for name, df in DATASETS.items()
])
display(summary)
display_info(f"<b>{len(DATASETS)}</b> dataset(s) satisfy the contract. "
             f"Next cell resolves their probe contracts.")



------------------------------------------------------------------------
STEP 2 / 6 — Processed dataset discovery
------------------------------------------------------------------------


## Contracts & probes

Two independent decisions are made here.

### 1. Contracts — what kind of label is this?

`_shared.contract_dict_for(dataset_name)` inspects the schema sidecar that
`master_dataset.py` wrote when the dataset was processed, and returns a
dict of `DatasetContract` fields. It resolves to one of four adapters:

| Adapter | Datasets | Target shape |
|---|---|---|
| `goemotions` | goemo | `(N, 28)` multi-hot |
| `isear` | isear | `(N,) int64`, 1..7 remapped to 0..6 |
| `dimensional` | (emobank) | `(N, D) float32` |
| `custom` | emotion, sst2, tweet_eval_emotion, amazon_polarity | `(N,) int64`, classes derived |

The resolver reads the `KNOWN_TARGET_TYPES` and `KNOWN_CLASS_ORDERS` dicts
at the bottom of `_shared.py`. Adding a new dataset needs no code change
here — the resolver picks it up from the sidecar.

### 2. Probes — how is the classifier shaped?

Four probes, in increasing capacity. Keeping the same four across every
`(model, dataset)` pair is what makes the layer curves comparable.

| Probe | Type | Complexity |
|---|---|---|
| `linear_logistic` | logistic | linear baseline; safest |
| `mlp_1_hidden`    | MLP | one hidden layer at 0.5·D |
| `mlp_2_hidden`    | MLP | + one at 0.25·D |
| `mlp_3_hidden`    | MLP | + one at 0.125·D |

Do not add a probe mid-matrix. The trial-folder hash depends on this list,
and changing it orphans whatever progress exists.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — CONTRACTS & PROBES
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 3 / 6 — Resolving contracts")

DATASET_CONTRACTS = {
    name: probe.DatasetContract(**_shared.contract_dict_for(name))
    for name in DATASETS
}

for name, c in DATASET_CONTRACTS.items():
    _say(
        f"  {name:22s} target={c.target_type:12s} "
        f"task={c.task_type:12s} "
        f"classes={len(c.class_order) if c.class_order else 'derived'}"
    )

_banner("STEP 3 / 6 — Probe specs")
probes = [
    probe.ProbeSpec(
        name="linear_logistic",
        type="logistic",
        complexity="linear",
        standardize=True,
        C=1.0,
        max_iter=3000,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_1_hidden",
        type="mlp",
        complexity="1_hidden",
        standardize=True,
        hidden_dims=["0.5d"],
        learning_rate=1e-3, weight_decay=1e-4,
        epochs=80, batch_size=256, patience=12,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_2_hidden",
        type="mlp",
        complexity="2_hidden",
        standardize=True,
        hidden_dims=["0.5d", "0.25d"],
        learning_rate=1e-3, weight_decay=1e-4,
        epochs=80, batch_size=256, patience=12,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_3_hidden",
        type="mlp",
        complexity="3_hidden",
        standardize=True,
        hidden_dims=["0.5d", "0.25d", "0.125d"],
        learning_rate=1e-3, weight_decay=1e-4,
        epochs=80, batch_size=256, patience=12,
        selection_metric="macro_f1",
    ),
]
_say(f"  {len(probes)} probe(s): {[p.name for p in probes]}")
display_info(f"<b>{len(DATASET_CONTRACTS)}</b> contract(s) resolved, "
             f"<b>{len(probes)}</b> probe(s) configured. "
             f"Next cell defines the on-disk discovery helpers.")


## Discovery helpers

Two small functions that walk the on-disk trees:

- `discover_extraction_pairs()` walks `hidden_states/<slug>/<dataset>/`,
  reads each `extraction.json`, and returns one row per complete artifact.
  This is the input to the probe matrix.

- `discover_probe_runs()` walks `interEx/<slug>/<dataset>/index.json` (the
  registry `Probe.py` maintains after each successful run) and returns one
  row per completed probe trial. Used by the post-run cells to reload
  results without re-fitting anything.

Both are defined here rather than in `Probe.py` because their only job is
to feed *this* notebook. `Probe.py` itself takes an explicit list of
entries, which keeps the fitting code free of filesystem assumptions.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — DISCOVERY HELPERS
# ═══════════════════════════════════════════════════════════════════════

def discover_extraction_pairs(hidden_states_root: Path = HIDDEN_STATES_ROOT) -> pd.DataFrame:
    """Every (model, dataset) with a completed extraction under hidden_states/."""
    rows = []
    if not hidden_states_root.is_dir():
        return pd.DataFrame(columns=["model", "dataset", "artifact_dir"])
    for model_dir in sorted(hidden_states_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            meta_path = dataset_dir / "extraction.json"
            if not (dataset_dir / "hidden_states.npy").is_file() or not meta_path.is_file():
                continue
            try:
                meta = json.loads(meta_path.read_text())
            except Exception:
                continue
            rows.append({
                "model":        meta.get("model", {}).get("name") or model_dir.name,
                "dataset":      meta.get("dataset", {}).get("name") or dataset_dir.name,
                "artifact_dir": str(dataset_dir),
            })
    return pd.DataFrame(rows)


def discover_probe_runs(interex_root: Path = INTEREX_ROOT) -> pd.DataFrame:
    """Every probe run registered under interEx/<slug>/<dataset>/index.json."""
    rows = []
    if not interex_root.is_dir():
        return pd.DataFrame(columns=["model", "dataset", "run_key", "trial_hash",
                                     "probes", "results_csv", "task_type", "n_classes"])
    for model_dir in sorted(interex_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            index = dataset_dir / "index.json"
            if not index.is_file():
                continue
            try:
                payload = json.loads(index.read_text())
            except Exception:
                continue
            for entry in payload.get("runs", []):
                results_csv = dataset_dir / entry["results_csv"]
                rows.append({
                    "model":       entry["model"],
                    "dataset":     entry["dataset"],
                    "run_key":     entry["run_key"],
                    "trial_hash":  entry["trial_hash"],
                    "probes":      "+".join(entry["probes"]),
                    "results_csv": str(results_csv),
                    "task_type":   entry["task_type"],
                    "n_classes":   entry["n_classes"],
                })
    return pd.DataFrame(rows)


_banner("STEP 4 / 6 — Discovery helpers ready")
_say(f"  HIDDEN_STATES_ROOT = {HIDDEN_STATES_ROOT}")
_say(f"  PROBE_ROOT         = {PROBE_ROOT}")
_say("  discover_extraction_pairs()  ->  reads hidden_states/")
_say("  discover_probe_runs()        ->  reads interEx/*/index.json")
print("\nNext cell enumerates the extraction artifacts available for probing.")


## Pre-flight

Everything below happens **before** a single probe is fitted. It is the
last chance to abort before a multi-hour run.

### The four checks

1. **Extraction artifacts present.** `discover_extraction_pairs()` must
   return at least one row. If it returns zero, extraction has not yet
   produced anything the probe can read.

2. **Every artifact has a contract.** Every dataset name in the discovery
   output must resolve via `DATASET_CONTRACTS`. Any dataset missing from
   the dict is dropped and reported — it will not appear in the matrix.

3. **Every artifact passes its contract.** Every extraction's metadata is
   checked against the contract's `task_type`. A single-task extraction
   (e.g. `isear`) misrouted through a multi-label contract would fail
   deeper, but we catch it here with a clearer message.

4. **Probe specs valid for every task type.** Every `ProbeSpec` must
   survive `Probe.validate_probe_spec` against every task type present in
   the target list. This is the fastest way to find a probe that will
   crash halfway through the matrix.

### Output

The next cell prints a target table — one row per `(model, dataset)` pair
that will be probed — and the total job count.

**Estimated time**: each probe × layer × repeat is one fit. A typical
combination has ~4 probes × 25 layers × 4 repeats = 400 fits, plus
~1,200 shuffled-label control fits. On CPU at ~1–3 seconds per fit, a
single `(model, dataset)` pair takes 20–40 minutes. Multiply by the
number of pairs in the target table.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — PRE-FLIGHT
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 5 / 6 — Pre-flight checks")

# ── 1. Any extraction artifacts at all? ───────────────────────────────
pairs = discover_extraction_pairs()
if pairs.empty:
    raise RuntimeError(
        "No extraction artifacts found under hidden_states/. "
        "Run __Extraction_Runner.ipynb first."
    )
_say(f"  OK  {len(pairs)} extraction artifact(s) on disk")

# ── 2. Every artifact has a contract ──────────────────────────────────
KNOWN_DATASETS = set(DATASET_CONTRACTS.keys())
known_pairs = pairs[pairs["dataset"].isin(KNOWN_DATASETS)].copy()
skipped     = pairs[~pairs["dataset"].isin(KNOWN_DATASETS)]
if not skipped.empty:
    _say(f"  !   {len(skipped)} artifact(s) skipped (no contract):")
    for _, r in skipped.head(8).iterrows():
        _say(f"        {r['model']:<45} {r['dataset']}")
if known_pairs.empty:
    raise RuntimeError("No (model, dataset) pairs resolved to a known contract.")

# ── 3. Contract task_type matches extraction metadata ─────────────────
_mismatches = []
for _, r in known_pairs.iterrows():
    try:
        meta = json.loads((Path(r["artifact_dir"]) / "extraction.json").read_text())
    except Exception:
        continue
    extraction_task = meta.get("extraction", {}).get("task_type")
    # metadata may not record task_type; only flag a positive mismatch
    contract_task = DATASET_CONTRACTS[r["dataset"]].task_type
    if extraction_task and extraction_task not in (contract_task, "auto", "unknown"):
        _mismatches.append((r["model"], r["dataset"], extraction_task, contract_task))
if _mismatches:
    for m, d, et, ct in _mismatches:
        _say(f"  X  {m}/{d}: extraction={et} contract={ct}")
    raise RuntimeError(f"{len(_mismatches)} task_type mismatch(es)")
_say("  OK  contract task_type matches every extraction")

# ── 4. Every probe spec valid for every task_type present ─────────────
_task_types = {DATASET_CONTRACTS[d].task_type for d in known_pairs["dataset"].unique()}
_spec_errors = []
for spec in probes:
    for tt in _task_types:
        try:
            probe.validate_probe_spec(spec, tt)
        except Exception as exc:
            _spec_errors.append(f"{spec.name} / {tt}: {exc}")
if _spec_errors:
    for e in _spec_errors:
        _say(f"  X  {e}")
    raise RuntimeError(f"{len(_spec_errors)} probe specification error(s)")
_say(f"  OK  all probes valid for task types {sorted(_task_types)}")

# ── Target table ──────────────────────────────────────────────────────
_banner("STEP 5 / 6 — Target list")

target_rows = []
for _, r in known_pairs.iterrows():
    c = DATASET_CONTRACTS[r["dataset"]]
    target_rows.append({
        "model":     r["model"].split("/")[-1],
        "dataset":   r["dataset"],
        "task_type": c.task_type,
        "classes":   len(c.class_order) if c.class_order else "derived",
    })
target_df = pd.DataFrame(target_rows).sort_values(["dataset", "model"])
display(target_df)

n_pairs = len(known_pairs)
n_jobs  = n_pairs * len(probes) * 25 * 4  # probes × layers × repeats (approx)
n_ctrl  = n_jobs * 3                       # shuffled_control_repeats
print()
_say(f"  Pairs to probe       : {n_pairs}")
_say(f"  Probes per pair      : {len(probes)}")
_say(f"  Approx. fits         : {n_jobs:,} main + {n_ctrl:,} control")
display_info(f"<b>Pre-flight passed.</b> Next cell launches the probe matrix.")


## Launch

The next cell calls `Probe.run_matrix` with:

| Parameter | Value | Meaning |
|---|---|---|
| `entries` | from cell 10 | one per `(model, dataset)` pair |
| `experiment_id` | `"main_run"` | recorded in every trial's metadata |
| `probes` | from cell 6 | the four classifiers |
| `repeats` | `4` | independent train/val/test splits |
| `max_samples` | `None` | use every sample in every dataset |
| `checkpoint_dir` | `PROBE_ROOT / "_matrix_checkpoint"` | resume support |
| `shuffled_label_control` | `True` | per-trial null |
| `shuffled_control_repeats` | `3` | three shuffles per probe-layer |

### Resume semantics

`run_matrix` writes a checkpoint JSON after every completed
`(model, dataset)` pair. If the notebook dies mid-run, re-executing this
cell picks up from the last completed pair — nothing is re-fitted.

### Progress

`Probe.py` uses `tqdm` for the per-trial progress bar. Interrupting the
kernel mid-trial is safe: the per-trial `progress.json` records each
completed `(repeat, layer, probe, control)` job, so the next call resumes
from the last finished fit.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 12 — RUN THE MATRIX
# ═══════════════════════════════════════════════════════════════════════

REPEATS        = 4
MAX_SAMPLES    = None
VERBOSE        = True
EXPERIMENT_ID  = "main_run"

_banner("STEP 6 / 6 — Launching probe matrix")

entries = []
for _, r in known_pairs.iterrows():
    entries.append({
        "model":        r["model"],
        "dataset":      r["dataset"],
        "artifact_dir": r["artifact_dir"],
        "contract":     DATASET_CONTRACTS[r["dataset"]],
        "dataset_df":   DATASETS[r["dataset"]],
    })

_say(f"  Pairs      : {len(entries)}")
_say(f"  Probes     : {len(probes)}")
_say(f"  Repeats    : {REPEATS}")
_say(f"  Checkpoint : {PROBE_ROOT / '_matrix_checkpoint'}")
print()

_t0 = time.perf_counter()

full_results = probe.run_matrix(
    entries,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=REPEATS,
    max_samples=MAX_SAMPLES,
    verbose=VERBOSE,
    checkpoint_dir=PROBE_ROOT / "_matrix_checkpoint",
    shuffled_label_control=True,
    shuffled_control_repeats=3,
)

_elapsed = time.perf_counter() - _t0

_banner("STEP 6 / 6 — Matrix finished")
_say(f"  Elapsed       : {_elapsed/60:.1f} min")
_say(f"  Result rows   : {len(full_results):,}")
_say(f"  Shape         : {full_results.shape}")
print("\nNext cell summarises and plots.")


## Post-run summary

The next cell rebuilds the analysis from disk. It works in a fresh kernel
too — it reloads everything it needs from `interEx/` and never depends on
cell 12 having run.

### What the outputs are

1. **Best-layer table** — the highest-Macro-F1 layer for every
   `(probe, model, dataset)` combination. This is the headline result.

2. **Cross-run plots** via `Probe.create_final_visuals`:
   - `layer_curve_*.png`   — mean across models per probe per metric
   - `heatmap_*.png`       — layer × probe grids
   - `final_probe_dashboard.png` — the four-panel summary

3. **Per-pair layer curves** — one figure per `(model, dataset)`, four
   probe lines each, so you can eyeball whether a specific pair peaks at
   a different depth than the average.

### Interpretation guardrails

- A `test_macro_f1` above **0.98** should trigger forensic review. It is
  not automatic leakage, but for this project it means running
  `probe_reporter.py` on that pair before quoting the number.
- The **selectivity gap** (`test_macro_f1 − control_macro_f1`) is the
  number that matters. F1 alone can be inflated by class imbalance;
  selectivity cannot.
- If different probes peak at *different* layers, the representation at
  the linear peak is not the same as the representation at the MLP peak.
  That is a result, not a bug.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 14 — POST-RUN SUMMARY
# ═══════════════════════════════════════════════════════════════════════

_banner("POST-RUN — Loading results")

# Prefer the in-memory frame; else reload from interEx/.
df = full_results if "full_results" in globals() and not full_results.empty else pd.DataFrame()
if df.empty:
    runs = discover_probe_runs()
    if not runs.empty:
        df = pd.concat(
            [pd.read_csv(p) for p in runs["results_csv"] if Path(p).is_file()],
            ignore_index=True,
        )
        _say(f"  Reloaded {len(df):,} rows from {runs['run_key'].nunique()} run(s)")
    else:
        _say("  No probe runs on disk yet.")
else:
    _say(f"  Using in-memory results: {len(df):,} rows")

if df.empty:
    display_info("Nothing to analyse yet. Run cell 12 first.")
else:
    # ── Best layer per (probe, model, dataset) ─────────────────────────
    _banner("POST-RUN — Best layer per probe")
    best_per_probe = (
        df.loc[df.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()]
          .sort_values(["dataset", "model", "probe"])
    )
    display(best_per_probe[[
        "probe", "model", "dataset", "layer_index",
        "test_macro_f1", "probe_score",
    ]])

    # ── Cross-model matrix ─────────────────────────────────────────────
    _banner("POST-RUN — Best Macro-F1 matrix")
    pivot_best = best_per_probe.pivot_table(
        index=["model", "dataset"], columns="probe", values="test_macro_f1",
    )
    display(pivot_best.style.background_gradient(cmap="viridis", axis=None))

    # ── Cross-run plots via Probe.create_final_visuals ─────────────────
    _banner("POST-RUN — Aggregate plots")
    plot_dir = Path("probe_plots")
    plot_dir.mkdir(exist_ok=True)
    probe.create_final_visuals(df, plot_dir)
    for p in sorted(plot_dir.glob("*.png")):
        _say(f"  {p.name}")

    # ── Per-pair layer curves ──────────────────────────────────────────
    _banner("POST-RUN — Per-pair layer curves")
    per_pair_dir = plot_dir / "per_pair"
    per_pair_dir.mkdir(parents=True, exist_ok=True)

    sns.set_style("darkgrid")
    plt.rcParams.update({
        "figure.facecolor": "#1e1e1e", "axes.facecolor": "#2d2d2d",
        "axes.edgecolor": "#d4d4d4",   "axes.labelcolor": "#d4d4d4",
        "text.color": "#d4d4d4",        "xtick.color": "#d4d4d4",
        "ytick.color": "#d4d4d4",       "grid.color": "#444444",
    })

    for (model, dataset), group in df.groupby(["model", "dataset"]):
        fig, ax = plt.subplots(figsize=(11, 5))
        for probe_name in sorted(group["probe"].unique()):
            sub = group[group["probe"] == probe_name].sort_values("layer_index")
            ax.plot(sub["layer_index"], sub["test_macro_f1"],
                    marker="o", linewidth=2, label=probe_name)
        ax.set_xlabel("Layer index")
        ax.set_ylabel("Test Macro-F1")
        ax.set_title(f"{model} / {dataset} — layer-wise Macro-F1")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9)
        fig.tight_layout()
        safe = f"{model.replace('/', '_')}__{dataset}.png"
        fig.savefig(per_pair_dir / safe, dpi=200, bbox_inches="tight")
        plt.close(fig)

    _say(f"  {len(list(per_pair_dir.glob('*.png')))} pair plot(s) written to {per_pair_dir}/")

    display_info(
        f"<b>Post-run complete.</b> "
        f"Aggregate plots in <code>{plot_dir}/</code>, "
        f"per-pair curves in <code>{per_pair_dir}/</code>. "
        f"For validity checks, run <code>python3 probe_reporter.py</code>."
    )
